In [2]:
from pyspark.sql.functions import (
    col, avg, min, max, count, to_date, when,
    sum as spark_sum, lag, unix_timestamp
)
from pyspark.sql.window import Window


StatementMeta(, cfc93fe5-b2f9-4ad4-ad06-c6c5dbbcdc18, 5, Finished, Available, Finished, False)

In [3]:
# CÉLULA 7 — Gold: Métricas Horárias
from pyspark.sql.functions import avg, min, max, count, to_date, sum as spark_sum
from pyspark.sql.window import Window
from pyspark.sql.functions import lag, unix_timestamp, expr

# Ler dados válidos do Silver
df_silver_valid = spark.sql("SELECT * FROM silver_sensors WHERE is_valid = true")

# Agregar métricas por sensor, dia e hora
df_hourly_metrics = df_silver_valid.groupBy(
    col("mac"),
    to_date(col("event_timestamp")).alias("event_date"),
    col("hour")
).agg(
    avg("temperature").alias("avg_temperature"),
    min("temperature").alias("min_temperature"),
    max("temperature").alias("max_temperature"),
    avg("humidity").alias("avg_humidity"),
    min("humidity").alias("min_humidity"),
    max("humidity").alias("max_humidity"),
    avg("battery").alias("avg_battery"),
    count("*").alias("readings_count")
).withColumn(
    "datetime",
    expr("to_timestamp(concat(event_date, ' ', lpad(hour, 2, '0'), ':00:00'))")
)

df_hourly_metrics.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold_hourly_metrics")
print(f"Gold hourly metrics: {df_hourly_metrics.count()} registos")

StatementMeta(, cfc93fe5-b2f9-4ad4-ad06-c6c5dbbcdc18, 6, Finished, Available, Finished, False)

Gold hourly metrics: 15984 registos


In [13]:
# CÉLULA 8 — Gold: Saúde dos Sensores

# Calcular intervalos entre leituras usando LAG
window_spec = Window.partitionBy("mac").orderBy("event_timestamp")

df_with_lag = df_silver_valid.withColumn(
    "prev_timestamp", lag("event_timestamp").over(window_spec)
).withColumn(
    "interval_seconds",
    unix_timestamp("event_timestamp") - unix_timestamp("prev_timestamp")
)

# Agregar saúde por sensor, dia e hora
df_sensor_health = df_with_lag.groupBy(
    col("mac"),
    to_date(col("event_timestamp")).alias("event_date"),
    col("hour")
).agg(
    count("*").alias("readings_count"),
    avg("interval_seconds").alias("avg_interval_seconds"),
    max("interval_seconds").alias("max_interval_seconds"),
    spark_sum(
        when(col("battery").isNull(), 1).otherwise(0)
    ).alias("null_battery_count"),
    spark_sum(
        when((col("interval_seconds") >= 600) & (col("interval_seconds") < 1800), 1).otherwise(0)
    ).alias("anomaly_e10_count"),
    spark_sum(
        when((col("interval_seconds") >= 1800) & (col("interval_seconds") < 3600), 1).otherwise(0)
    ).alias("anomaly_e30_count"),
    spark_sum(
        when(col("interval_seconds") >= 3600, 1).otherwise(0)
    ).alias("anomaly_e60_count")
).withColumn(
    "datetime",
    expr("to_timestamp(concat(event_date, ' ', lpad(hour, 2, '0'), ':00:00'))")
)

df_sensor_health.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold_sensor_health")
print(f"Gold sensor health: {df_sensor_health.count()} registos")

# Agregar leituras diárias por sensor
df_daily_readings = df_sensor_health.groupBy(
    "mac",
    "event_date"
).agg(
    spark_sum("readings_count").alias("daily_readings_count")
)

df_daily_readings.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold_daily_readings")
print(f"Gold daily readings: {df_daily_readings.count()} registos")

StatementMeta(, cfc93fe5-b2f9-4ad4-ad06-c6c5dbbcdc18, 47, Finished, Available, Finished, False)

Gold sensor health: 15984 registos
Gold daily readings: 718 registos


In [18]:
df = spark.sql("""
    SELECT 
        mac,
        COUNT(*) as total_horas,
        SUM(CASE WHEN anomaly_e10_count > 0 THEN 1 ELSE 0 END) as horas_com_e10,
        ROUND(SUM(CASE WHEN anomaly_e10_count > 0 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) as pct_horas_com_e10
    FROM gold_sensor_health
    WHERE event_date >= '2026-04-01'
    GROUP BY mac
    ORDER BY mac
""")

df.show()

StatementMeta(, cfc93fe5-b2f9-4ad4-ad06-c6c5dbbcdc18, 62, Finished, Available, Finished, False)

+-----------------+-----------+-------------+-----------------+
|              mac|total_horas|horas_com_e10|pct_horas_com_e10|
+-----------------+-----------+-------------+-----------------+
|A4:C1:38:06:A5:98|        461|           34|             7.38|
|A4:C1:38:0E:16:F8|        461|           34|             7.38|
|A4:C1:38:7F:F4:87|        461|           34|             7.38|
|A4:C1:38:85:5E:41|        461|           34|             7.38|
|A4:C1:38:98:9E:DA|        461|           34|             7.38|
+-----------------+-----------+-------------+-----------------+

